In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # ARC LAB — Transform: Bronze → Silver
# MAGIC **Cleans, deduplicates, and quality-flags raw WattTime data.**
# MAGIC
# MAGIC - Filters to `lbs_co2_per_mwh` records only (drops `percentile` real-time records)
# MAGIC - Deduplicates on 5-minute bucket
# MAGIC - Adds local timestamp (America/Los_Angeles)
# MAGIC - Applies quality flags (range check + model transition flag)
# MAGIC - Overwrites Silver table on each run (idempotent)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Config

# COMMAND ----------

from pyspark.sql import functions as F
from datetime import datetime, timezone

CATALOG       = "bootcamp_students"
BRONZE_TABLE  = f"{CATALOG}.bronze.watttime_raw"
SILVER_TABLE  = f"{CATALOG}.silver.watttime_clean"
LOCAL_TZ      = "America/Los_Angeles"
MODEL_CHANGE  = "2026-03-18"   # WattTime model update date

print(f"Source : {BRONZE_TABLE}")
print(f"Target : {SILVER_TABLE}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Read Bronze

# COMMAND ----------

bronze = spark.table(BRONZE_TABLE)
print(f"✓ Bronze records loaded : {bronze.count():,}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Filter to lbs_co2_per_mwh Only
# MAGIC The real-time endpoint returns `percentile` units — those are not comparable
# MAGIC to the historical `lbs_co2_per_mwh` records and are excluded from Silver.

# COMMAND ----------

bronze_filtered = bronze.filter(F.col("units") == "lbs_co2_per_mwh")
print(f"✓ After units filter    : {bronze_filtered.count():,} records")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Deduplicate on 5-Minute Bucket
# MAGIC WattTime returns one reading per 5-minute interval. If Bronze was appended
# MAGIC multiple times (e.g. backfill + re-run), this ensures one clean record per slot.

# COMMAND ----------

bronze_deduped = bronze_filtered.withColumn(
    "ts_5min_bucket",
    (
        F.floor(F.unix_timestamp(F.col("ts_utc").cast("timestamp")) / 300) * 300
    ).cast("timestamp").cast("timestamp_ntz")
).dropDuplicates(["region_code", "signal_type", "ts_5min_bucket"])

print(f"✓ After dedup           : {bronze_deduped.count():,} records")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Add Timestamps and Quality Flags

# COMMAND ----------

processed_at = datetime.now(timezone.utc).replace(tzinfo=None)

silver = bronze_deduped \
    .withColumn(
        "ts_local",
        F.from_utc_timestamp(F.col("ts_utc").cast("timestamp"), LOCAL_TZ).cast("timestamp_ntz")
    ) \
    .withColumn("ts_date", F.to_date(F.col("ts_utc"))) \
    .withColumn(
        "lbs_co2_per_mwh", F.col("value")
    ) \
    .withColumn(
        "range_flag",
        (F.col("value") < 0) | (F.col("value") > 2000)
    ) \
    .withColumn(
        "quality_flag",
        F.when((F.col("value") < 0) | (F.col("value") > 2000), F.lit("WARN"))
         .otherwise(F.lit("PASS"))
    ) \
    .withColumn(
        "model_transition_flag",
        F.to_date(F.col("ts_utc")) >= F.lit(MODEL_CHANGE)
    ) \
    .withColumn("processed_at", F.lit(processed_at).cast("timestamp_ntz"))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Select Final Columns

# COMMAND ----------

silver_final = silver.select(
    "region_code",
    "signal_type",
    "ts_utc",
    "ts_local",
    "ts_date",
    "ts_5min_bucket",
    "lbs_co2_per_mwh",
    "model_version",
    "model_transition_flag",
    "range_flag",
    "quality_flag",
    "ingested_at",
    "processed_at"
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Write Silver Table

# COMMAND ----------

silver_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("ts_date") \
    .saveAsTable(SILVER_TABLE)

count = spark.table(SILVER_TABLE).count()
print(f"✓ Silver complete : {count:,} records written to {SILVER_TABLE}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8. Quality Check

# COMMAND ----------

print("=== QUALITY FLAG SUMMARY ===")
display(spark.sql(f"""
    SELECT
        quality_flag,
        COUNT(*)                          AS record_count,
        ROUND(MIN(lbs_co2_per_mwh), 2)   AS min_intensity,
        ROUND(MAX(lbs_co2_per_mwh), 2)   AS max_intensity,
        ROUND(AVG(lbs_co2_per_mwh), 2)   AS avg_intensity
    FROM {SILVER_TABLE}
    GROUP BY quality_flag
    ORDER BY quality_flag
"""))

# COMMAND ----------

print("=== DATE RANGE ===")
display(spark.sql(f"""
    SELECT
        MIN(ts_utc) AS earliest,
        MAX(ts_utc) AS latest,
        COUNT(DISTINCT ts_date) AS days_covered,
        COUNT(*) AS total_records
    FROM {SILVER_TABLE}
"""))

Source : bootcamp_students.bronze.watttime_raw
Target : bootcamp_students.silver.watttime_clean
✓ Bronze records loaded : 2,018
✓ After units filter    : 2,016 records
✓ After dedup           : 2,016 records
✓ Silver complete : 2,016 records written to bootcamp_students.silver.watttime_clean
=== QUALITY FLAG SUMMARY ===


quality_flag,record_count,min_intensity,max_intensity,avg_intensity
PASS,2016,35.0,1089.0,721.31


=== DATE RANGE ===


earliest,latest,days_covered,total_records
2026-02-27T23:20:00.000,2026-03-06T23:15:00.000,8,2016
